# Imputation of Daily Feature Zarr Arrays

## Objective
This notebook performs missing-value imputation for the daily feature Zarr arrays stored in `FEATURE/feature_zarr`. The procedure operates feature by feature and day by day, producing an imputed feature archive together with summary statistics describing the amount of missingness before and after imputation.

## Imputation Strategy
The default configuration applies local spatial interpolation with a `3 x 3` kernel and uses raster-based filling when available. A nearest-neighbor fallback is retained so that residual missing cells can be resolved when the local interpolation stage is insufficient.

## Outputs
The notebook writes:
- imputed feature Zarr arrays
- per-feature summary statistics
- per-day imputation statistics

These outputs are used as the cleaned feature source for downstream historical-window feature construction.


## Environment and Imports

This section loads the imputation utilities and the basic analysis libraries used for configuration, execution, and inspection of the generated statistics.


In [1]:
from pathlib import Path
import json
import pandas as pd

from zarr_feature_imputer import ImputeConfig, run_imputation


## Configuration

This section defines the source feature archive, the destination directories, and the imputation settings. These parameters determine how each daily feature grid is processed and how the resulting diagnostics are stored.


In [2]:
# -------------------------
# Config
# -------------------------
BASE = Path('/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE')
SRC_DIR = BASE / 'feature_zarr'
OUT_DIR = BASE / 'imputed_feature_zarr'
STATS_DIR = BASE / 'imputed_feature_zarr_stats'

cfg = ImputeConfig(
    source_dir=SRC_DIR,
    output_dir=OUT_DIR,
    stats_dir=STATS_DIR,
    include_features=None,
    exclude_features=(),
    kernel_size=3,
    max_passes=1,
    use_rasterio_fillnodata=True,
    rasterio_max_search_distance=100.0,
    rasterio_smoothing_iterations=0,
    nearest_fill_fallback=True,
    ensure_no_nan=True,
    n_jobs=10,
    show_progress=True,
    copy_manifest_if_present=True,
)

cfg


ImputeConfig(source_dir=PosixPath('/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/feature_zarr'), output_dir=PosixPath('/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/imputed_feature_zarr'), stats_dir=PosixPath('/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/imputed_feature_zarr_stats'), include_features=None, exclude_features=(), kernel_size=3, max_passes=1, use_rasterio_fillnodata=True, rasterio_max_search_distance=100.0, rasterio_smoothing_iterations=0, nearest_fill_fallback=True, ensure_no_nan=True, n_jobs=10, show_progress=True, copy_manifest_if_present=True)

## Imputation Run

The configured imputation pipeline is executed here. The returned output object records the paths of the generated Zarr archive and the diagnostic summary files.


In [3]:
# -------------------------
# Run imputation
# -------------------------
out = run_imputation(cfg)
print(json.dumps(out, indent=2))


features(parallel):   0%|          | 0/25 [00:00<?, ?feat/s]/Users/copperhead07/Computing/DataCollection/RealWork/.venv/lib/python3.14/site-packages/rasterio/fill.py:72: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  return _fillnodata(image, mask, max_search_distance, smoothing_iterations, **filloptions)

























































































































































































































































































































































































































































































































































































































































{
  "summary_csv": "/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/imputed_feature_zarr_stats/feature_impute_summary.csv",
  "summary_json": "/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/imputed_feature_zarr_stats/feature_impute_summary.json",
  "day_stats_csv": "/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/imputed_feature_zarr_stats/feature_impute_day_stats.csv",
  "run_metadata": "/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/imputed_feature_zarr_stats/impute_run_metadata.json",
  "output_dir": "/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/imputed_feature_zarr"
}


## Summary Inspection

This section inspects the per-feature summary table, which reports the total amount of missingness before and after imputation and provides a compact view of feature-level data quality.


In [4]:
# -------------------------
# Inspect per-feature stats
# -------------------------
summary = pd.read_csv(out['summary_csv'])
summary = summary.sort_values(['nan_before_total', 'feature'], ascending=[False, True]).reset_index(drop=True)
summary.head(30)


,feature,source,output,kind,shape,kernel,max_passes,cells_total,nan_before_total,nan_after_total,...,filled_kernel_total,filled_rasterio_total,filled_nearest_total,filled_global_total,nearest_iters_total,used_rasterio,fill_rate_pct_of_nan_before,days_with_nan_before,days_with_nan_after,days_changed
0,ignition_prob_clim,/Users/copperhead07/Computing/DataCollection/R...,/Users/copperhead07/Computing/DataCollection/R...,dynamic,"[4363, 142, 116]",3,1,71867336,67719124,0,...,24865910,42853214,0,0,0,True,100.0,4363,0,4363
1,aet,/Users/copperhead07/Computing/DataCollection/R...,/Users/copperhead07/Computing/DataCollection/R...,dynamic,"[4363, 142, 116]",3,1,71867336,61318752,0,...,2023912,59294840,0,0,0,True,100.0,4363,0,4363
2,bui,/Users/copperhead07/Computing/DataCollection/R...,/Users/copperhead07/Computing/DataCollection/R...,dynamic,"[4363, 142, 116]",3,1,71867336,61318752,0,...,2023912,59294840,0,0,0,True,100.0,4363,0,4363
3,burn_idx,/Users/copperhead07/Computing/DataCollection/R...,/Users/copperhead07/Computing/DataCollection/R...,dynamic,"[4363, 142, 116]",3,1,71867336,61318752,0,...,2023912,59294840,0,0,0,True,100.0,4363,0,4363
4,dc,/Users/copperhead07/Computing/DataCollection/R...,/Users/copperhead07/Computing/DataCollection/R...,dynamic,"[4363, 142, 116]",3,1,71867336,61318752,0,...,2023912,59294840,0,0,0,True,100.0,4363,0,4363
5,dmc,/Users/copperhead07/Computing/DataCollection/R...,/Users/copperhead07/Computing/DataCollection/R...,dynamic,"[4363, 142, 116]",3,1,71867336,61318752,0,...,2023912,59294840,0,0,0,True,100.0,4363,0,4363
6,erc,/Users/copperhead07/Computing/DataCollection/R...,/Users/copperhead07/Computing/DataCollection/R...,dynamic,"[4363, 142, 116]",3,1,71867336,61318752,0,...,2023912,59294840,0,0,0,True,100.0,4363,0,4363
7,ffmc,/Users/copperhead07/Computing/DataCollection/R...,/Users/copperhead07/Computing/DataCollection/R...,dynamic,"[4363, 142, 116]",3,1,71867336,61318752,0,...,2023912,59294840,0,0,0,True,100.0,4363,0,4363
8,fm100,/Users/copperhead07/Computing/DataCollection/R...,/Users/copperhead07/Computing/DataCollection/R...,dynamic,"[4363, 142, 116]",3,1,71867336,61318752,0,...,2023912,59294840,0,0,0,True,100.0,4363,0,4363
9,fwi,/Users/copperhead07/Computing/DataCollection/R...,/Users/copperhead07/Computing/DataCollection/R...,dynamic,"[4363, 142, 116]",3,1,71867336,61318752,0,...,2023912,59294840,0,0,0,True,100.0,4363,0,4363


## Day-Level Inspection

The final section examines per-day imputation statistics for a selected feature. This provides a finer-grained view of when missing-value correction was required within the daily time series.


In [5]:
# -------------------------
# Inspect per-day stats for one feature
# -------------------------
day_stats = pd.read_csv(out['day_stats_csv'])
FEATURE_NAME = 'aet'
day_stats.query('feature == @FEATURE_NAME').head(20)


,feature,day_index,nan_before,nan_after,filled,changed,filled_kernel,filled_rasterio,filled_nearest,filled_global,nearest_iters
26181,aet,0,16472,0,16472,1,0,16472,0,0,0
26182,aet,1,16472,0,16472,1,0,16472,0,0,0
26183,aet,2,16472,0,16472,1,0,16472,0,0,0
26184,aet,3,16472,0,16472,1,0,16472,0,0,0
26185,aet,4,4375,0,4375,1,2321,2054,0,0,0
26186,aet,5,16472,0,16472,1,0,16472,0,0,0
26187,aet,6,16472,0,16472,1,0,16472,0,0,0
26188,aet,7,16472,0,16472,1,0,16472,0,0,0
26189,aet,8,16472,0,16472,1,0,16472,0,0,0
26190,aet,9,4375,0,4375,1,2321,2054,0,0,0
